In [31]:
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, START, END
import random

In [47]:
class AgentState(TypedDict):
    name: str
    guess: List[int]
    ctr: int
    input: int
    lower_bound: int
    upper_bound: int

In [54]:
def setup_node(state: AgentState) -> AgentState:
    """Setup node"""
    state['name'] = f"Hello {state['name']}"
    state['ctr'] = 0
    state['guess'] = []

    return state

def generate_rand(state: AgentState) -> AgentState:
    """Random no generator"""
    state['guess'].append(random.randint(state['lower_bound'], state['upper_bound']))

    return state

def hint_node(state: AgentState) -> AgentState:
    """Hints node"""
    if state['guess'][-1] > state['input']:
        print('Lower!')
    else:
        print('Higher')

    return state

def decision_node(state: AgentState):
    """Decides if the loop should continue"""
    if state['input'] in state['guess'] or len(state['guess']) > 7:
        return "no"
    else:
        return "yes"

In [55]:
graph = StateGraph(AgentState)

graph.add_node("setup", setup_node)
graph.add_node("guesser", generate_rand)
graph.add_node("hint", hint_node)
#graph.add_node("looper", decision_node)

graph.set_entry_point("setup")
graph.add_edge("setup", "guesser")
graph.add_edge("guesser", "hint")
graph.add_conditional_edges(
    "hint",
    decision_node,
    {
        "no": END,
        "yes": "guesser"
    }
)
app = graph.compile()

In [56]:
res = app.invoke({'name':'kv', 'lower_bound': 5, 'upper_bound': 100,
                  'input': 62})
res

Lower!
Higher
Higher
Higher
Higher
Lower!
Higher
Higher


{'name': 'Hello kv',
 'guess': [78, 37, 8, 32, 7, 90, 7, 59],
 'ctr': 0,
 'input': 62,
 'lower_bound': 5,
 'upper_bound': 100}